# AWS sensor-fault detector — general model training

Trains **one** Isolation Forest for the whole station network and validates it.

**How to run this:** drop the training CSVs into `ml/data/` (git-ignored — not shipped
with the repo), then run this notebook top to bottom. It reads `ml/data/clean_data/`,
trains, validates, and writes `ml/models/general_model.joblib`, which the FastAPI
service loads.

**What the model does:** for each reading it flags whether a sensor is transmitting
faulty data — spike, drift, jitter (learned) or stuck-at / impossible rainfall
(deterministic rules). It is *detection*, not forecasting.

**Read before trusting the numbers:** the network has no field-verified fault labels,
so recall is measured against *synthetic* injected faults and false-positive rate
against genuine clean holdout. Treat recall as "detects faults shaped like these"; the
FPR (measured on real clean data) is the trustworthy half.


In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, os, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(".."))     # import ml/aws_anomaly.py
import aws_anomaly as A

DATA  = "../data/clean_data"
MODEL = "../models/general_model.joblib"

# Size/quality config chosen from the tradeoff sweep (see README).
N_ESTIMATORS = 200
MAX_SAMPLES  = 50_000
print("stations found:", sum(1 for _ in A.iter_station_files(DATA)))


stations found: 13


## Train the general model (all stations pooled)


In [2]:
bundle, report = A.train_general(DATA, n_estimators=N_ESTIMATORS, max_samples=MAX_SAMPLES)
print("sensors:", bundle["sensors"])
print("per-sensor thresholds:", {k: round(v, 3) for k, v in bundle["thresholds"].items()})


sensors: ['wind_speed', 'wind_dir', 'solar', 'air_temp', 'humidity', 'pressure', 'soil_temp', 'soil_moist']
per-sensor thresholds: {'wind_speed': -0.029, 'wind_dir': 0.054, 'solar': 0.074, 'air_temp': -0.056, 'humidity': -0.043, 'pressure': -0.035, 'soil_temp': -0.034, 'soil_moist': -0.019}


## Validation — per sensor, from the single shared model


In [3]:
cols = ["sensor", "n_train", "clean_fpr", "recall", "precision", "f1", "roc_auc",
        "pr_auc", "recall_spike", "recall_drift", "recall_jitter", "recall_stuck"]
report[cols].round(3)


,sensor,n_train,clean_fpr,recall,precision,f1,roc_auc,pr_auc,recall_spike,recall_drift,recall_jitter,recall_stuck
0,wind_speed,279468,0.010,0.510,0.777,0.616,0.862,0.615,1.0,0.038,1.0,0.003
1,wind_dir,121558,0.009,0.346,0.830,0.488,0.684,0.565,1.0,0.036,NaN,0.000
2,solar,166663,0.009,0.508,0.859,0.638,0.894,0.682,1.0,0.030,1.0,0.002
3,air_temp,231676,0.010,0.692,0.844,0.761,0.911,0.766,1.0,0.766,1.0,0.003
4,humidity,233067,0.009,0.526,0.820,0.641,0.918,0.698,1.0,0.098,1.0,0.005
5,pressure,231865,0.009,0.605,0.840,0.704,0.948,0.737,1.0,0.415,1.0,0.005
6,soil_temp,205747,0.008,0.595,0.862,0.704,0.909,0.747,1.0,0.380,1.0,0.002
7,soil_moist,195084,0.018,0.506,0.722,0.595,0.809,0.568,1.0,0.024,1.0,0.001


In [4]:
rx = report[["recall_spike", "recall_drift", "recall_jitter"]].mean(axis=1)
print(f"MEAN  clean_fpr={report.clean_fpr.mean():.4f}   "
      f"recall(excl stuck)={rx.mean():.3f}   roc_auc={report.roc_auc.mean():.3f}   "
      f"precision={report.precision.mean():.3f}")
print()
print("Stuck-at is ~0 for the model by design and is caught by the deterministic")
print("rule instead — a frozen sensor sits at a perfectly normal value.")


MEAN  clean_fpr=0.0101   recall(excl stuck)=0.721   roc_auc=0.867   precision=0.819

Stuck-at is ~0 for the model by design and is caught by the deterministic
rule instead — a frozen sensor sits at a perfectly normal value.


## Recall by fault type — where the model is strong vs weak


In [5]:
fa = ["recall_spike", "recall_drift", "recall_jitter", "recall_stuck"]
bt = report.set_index("sensor")[fa]
bt.columns = [c.replace("recall_", "") for c in bt.columns]
ax = bt.plot.bar(figsize=(11, 4.5), rot=0, width=0.82)
ax.set_ylim(0, 1.05); ax.set_ylabel("recall at ~1% FPR")
ax.set_title("Recall by fault type and sensor")
ax.legend(title=None, ncol=4); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [6]:
ax = report.set_index("sensor")[["clean_fpr"]].plot.bar(
    figsize=(9, 3.8), rot=0, legend=False, color="#c0392b")
ax.axhline(A.CONTAM, ls="--", c="k", lw=1, label=f"target {A.CONTAM:.0%}")
ax.set_ylabel("false-positive rate")
ax.set_title("False-positive rate per sensor (lower is better)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


## Save the model


In [7]:
A.save_bundle(bundle, MODEL)
print(f"saved {MODEL}  ({os.path.getsize(MODEL)/1e6:.1f} MB)")
report.to_csv("../models/general_validation.csv", index=False)


saved ../models/general_model.joblib  (25.8 MB)


## Smoke test: score a window the way the live service will

Builds a dense 6.5-hour window and runs it through the exact serving function the
FastAPI service calls, so this exercises the real code path — including injected faults.


In [8]:
ts = pd.date_range('2024-01-01', periods=80, freq='5min')
rng = np.random.default_rng(1)
b = lambda mu, sd: mu + rng.normal(0, sd, 80)
win = pd.DataFrame({'timestamp': ts,
    'temperature': b(25, .3), 'humidity': b(70, 1.), 'pressure': b(900, .2),
    'wind_speed': np.abs(b(1., .4)), 'wind_direction': b(180, 15) % 360,
    'soil_moisture': b(20, .5), 'rain': np.zeros(80)})

print('CLEAN window:')
for k, v in A.score_window(bundle, win, 'TEST').items():
    print('  ', k, 'flag=', v['flag'], v['reason'])

w = win.copy(); w.loc[79, 'temperature'] = 60.0
print('spike temp -> 60C:', A.score_window(bundle, w, 'TEST').get('air_temp'))
w = win.copy(); w['humidity'] = 70.0
print('stuck humidity:', A.score_window(bundle, w, 'TEST').get('humidity'))
w = win.copy(); w.loc[79, 'rain'] = 40.0
print('impossible rain:', A.score_window(bundle, w, 'TEST').get('rainfall'))


CLEAN window:
   air_temp flag= 0 ok
   humidity flag= 0 ok
   pressure flag= 0 ok
   wind_speed flag= 0 ok
   wind_dir flag= 0 ok
   soil_moist flag= 1 anomaly
   rainfall flag= 0 ok


spike temp -> 60C: {'flag': 1, 'score': 0.1924, 'reason': 'anomaly'}
stuck humidity: {'flag': 1, 'score': -0.0617, 'reason': 'stuck'}


impossible rain: {'flag': 1, 'score': None, 'reason': 'rain_rate'}


## Notes / limitations

- **One model, all stations.** When real stations with history arrive, a per-station
  model can be trained and the serving layer can prefer it; this general model stays
  as the fallback.
- **Train vs serve gaps** (documented in `aws_anomaly.py`, unresolvable until the real
  stations and firmware are known): units (calibrated vs raw), cadence (5-min vs
  ~15-min live), and within-bin std (absent live). Per-sensor robust scaling absorbs
  linear unit differences but not a nonlinear calibration curve.
- **Drift recall is weak** (0.02–0.77) because every feature scores a row independently;
  a sequence model is the natural next step if drift matters.
- **Rainfall** is rule-based, not learned. Note the known upstream units bug at the
  seven 15-second-cadence stations (see the main project session log).
